# E-commerce Sales Data Analysis

This notebook analyzes e-commerce order data to identify revenue trends, profitability drivers, customer behavior, and operational opportunities. It is structured as a portfolio-ready data analysis project with clean data preparation, exploratory analysis, RFM segmentation, SQL analysis, and business recommendations.


## SECTION 1 — IMPORTS AND SETUP


In [ ]:
from pathlib import Path
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda value: f'{value:.2f}')
warnings.filterwarnings('ignore')

COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']

DATA_PATH = 'data/raw/ecommerce_sales.csv'
CLEANED_DATA_PATH = 'data/cleaned/cleaned_orders.csv'
CHARTS_DIR = Path('outputs/charts')
REPORTS_DIR = Path('outputs/reports')

CHARTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## SECTION 2 — LOAD DATA


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns')
print()
print('Column data types:')
print(df.dtypes)

print()
print('First 5 rows:')
display(df.head())

print()
print('DataFrame summary:')
df.info()


## SECTION 3 — DATA CLEANING


### Step 3.1 — Duplicate removal


In [ ]:
# Standardized names make the notebook easier to read and keep SQL queries portable.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

duplicate_count_before = df.duplicated().sum()
row_count_before = df.shape[0]
df = df.drop_duplicates().copy()
duplicate_count_after = df.duplicated().sum()

print(f'Duplicate rows before removal: {duplicate_count_before:,}')
print(f'Rows before duplicate removal: {row_count_before:,}')
print(f'Duplicate rows after removal: {duplicate_count_after:,}')
print(f'Rows after duplicate removal: {df.shape[0]:,}')


### Step 3.2 — Missing values


In [ ]:
print('Missing values before cleaning:')
print(df.isnull().sum())

critical_value_columns = ['sales', 'profit']
shipping_cost_median = df['shipping_cost'].median()

df = df.dropna(subset=critical_value_columns).copy()
df['shipping_cost'] = df['shipping_cost'].fillna(shipping_cost_median)
df['discount'] = df['discount'].fillna(0)

print()
print('Missing values after targeted fixes:')
print(df.isnull().sum())


### Step 3.3 — Data type fixes


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

string_columns = df.select_dtypes(include='object').columns
df[string_columns] = df[string_columns].apply(lambda column: column.str.strip())

print('Updated column data types:')
print(df.dtypes)


### Step 3.4 — Derived columns


In [ ]:
SMALL_ORDER_MAX_QUANTITY = 2
MEDIUM_ORDER_MAX_QUANTITY = 5

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_year'] = df['order_date'].dt.to_period('M')
df['profit_margin'] = np.where(df['sales'] != 0, (df['profit'] / df['sales']) * 100, np.nan).round(2)
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days

df['order_size'] = np.select(
    [
        df['quantity'] <= SMALL_ORDER_MAX_QUANTITY,
        df['quantity'].between(SMALL_ORDER_MAX_QUANTITY + 1, MEDIUM_ORDER_MAX_QUANTITY),
        df['quantity'] > MEDIUM_ORDER_MAX_QUANTITY,
    ],
    ['Small', 'Medium', 'Large'],
    default='Unknown'
)

print('Derived columns created successfully.')
display(df[['order_date', 'ship_date', 'year', 'month', 'month_year', 'profit_margin', 'days_to_ship', 'order_size']].head())


### Step 3.5 — Sanity check


In [ ]:
print(f'Final cleaned shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns')
print()
print('Null counts after cleaning:')
print(df.isnull().sum())

df.to_csv(CLEANED_DATA_PATH, index=False)
print(f'Cleaned data exported to: {CLEANED_DATA_PATH}')


## SECTION 4 — EXPLORATORY DATA ANALYSIS


### 4.1 — Summary KPIs


In [ ]:
total_revenue = df['sales'].sum()
total_profit = df['profit'].sum()
overall_profit_margin = (total_profit / total_revenue) * 100
total_orders = df['order_id'].nunique()
average_order_value = total_revenue / total_orders
average_days_to_ship = df['days_to_ship'].mean()

print(f'Total Revenue: ${total_revenue:,.2f}')
print(f'Total Profit: ${total_profit:,.2f}')
print(f'Overall Profit Margin: {overall_profit_margin:.2f}%')
print(f'Total Orders: {total_orders:,}')
print(f'Average Order Value: ${average_order_value:,.2f}')
print(f'Average Days to Ship: {average_days_to_ship:.2f} days')


### 4.2 — Monthly Sales Trend


In [ ]:
monthly_sales = df.groupby('month_year', as_index=False)['sales'].sum()
monthly_sales['month_year'] = monthly_sales['month_year'].astype(str)
mean_monthly_sales = monthly_sales['sales'].mean()

plt.figure(figsize=(10, 5))
ax = plt.gca()
ax.plot(monthly_sales['month_year'], monthly_sales['sales'], color=COLORS[0], marker='o', markersize=4, linewidth=2)
ax.fill_between(monthly_sales['month_year'], monthly_sales['sales'], alpha=0.1, color=COLORS[0])
ax.axhline(mean_monthly_sales, color='red', linestyle='--', linewidth=1.5, label=f'Mean: ${mean_monthly_sales:,.0f}')
ax.set_title('Monthly Sales Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Sales ($)', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.3 — Sales by Category


In [ ]:
category_performance = df.groupby('category', as_index=True)[['sales', 'profit']].sum().sort_values('sales', ascending=True)
bar_height = 0.35
category_positions = np.arange(len(category_performance))

plt.figure(figsize=(10, 5))
ax = plt.gca()
sales_bars = ax.barh(category_positions - bar_height / 2, category_performance['sales'], height=bar_height, color=COLORS[0], label='Sales')
profit_bars = ax.barh(category_positions + bar_height / 2, category_performance['profit'], height=bar_height, color=COLORS[1], label='Profit')
ax.set_yticks(category_positions)
ax.set_yticklabels(category_performance.index, fontsize=10)
ax.bar_label(sales_bars, fmt='$%.0f', padding=3, fontsize=9)
ax.bar_label(profit_bars, fmt='$%.0f', padding=3, fontsize=9)
ax.set_title('Sales and Profit by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Amount ($)', fontsize=11)
ax.set_ylabel('Category', fontsize=11)
ax.tick_params(axis='x', labelsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/sales_by_category.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.4 — Top 10 Products by Revenue


In [ ]:
TOP_PRODUCT_COUNT = 10

top_products = (
    df.groupby('product_name', as_index=False)['sales']
    .sum()
    .nlargest(TOP_PRODUCT_COUNT, 'sales')
    .sort_values('sales', ascending=True)
)

plt.figure(figsize=(10, 5))
ax = plt.gca()
bars = ax.barh(top_products['product_name'], top_products['sales'], height=0.6, color=COLORS[0])
ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=9)
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)', fontsize=11)
ax.set_ylabel('Product Name', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/charts/top_10_products_by_revenue.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.5 — Regional Performance


In [ ]:
regional_performance = (
    df.groupby('region', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), profit_margin=('profit_margin', 'mean'))
    .sort_values('sales', ascending=False)
)

bar_width = 0.35
region_positions = np.arange(len(regional_performance))

plt.figure(figsize=(10, 5))
ax = plt.gca()
sales_bars = ax.bar(region_positions - bar_width / 2, regional_performance['sales'], width=bar_width, color=COLORS[0], label='Sales')
profit_bars = ax.bar(region_positions + bar_width / 2, regional_performance['profit'], width=bar_width, color=COLORS[1], label='Profit')
ax.bar_label(sales_bars, fmt='$%.0f', padding=3, fontsize=9)
ax.bar_label(profit_bars, fmt='$%.0f', padding=3, fontsize=9)
ax.set_xticks(region_positions)
ax.set_xticklabels(regional_performance['region'], fontsize=10)
ax.set_title('Regional Sales and Profit Performance', fontsize=14, fontweight='bold')
ax.set_xlabel('Region', fontsize=11)
ax.set_ylabel('Amount ($)', fontsize=11)
ax.tick_params(axis='y', labelsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/charts/regional_performance.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.6 — Profit Margin by Sub-Category


In [ ]:
HIGH_MARGIN_THRESHOLD = 15
BREAK_EVEN_MARGIN = 0

subcategory_margin = (
    df.groupby('sub_category', as_index=False)['profit_margin']
    .mean()
    .sort_values('profit_margin', ascending=True)
)
bar_colors = np.where(
    subcategory_margin['profit_margin'] < BREAK_EVEN_MARGIN,
    'red',
    np.where(subcategory_margin['profit_margin'] > HIGH_MARGIN_THRESHOLD, 'green', 'steelblue')
)

plt.figure(figsize=(10, 5))
ax = plt.gca()
bars = ax.barh(subcategory_margin['sub_category'], subcategory_margin['profit_margin'], height=0.6, color=bar_colors)
ax.bar_label(bars, fmt='%.2f%%', padding=3, fontsize=9)
ax.axvline(BREAK_EVEN_MARGIN, color='black', linewidth=1)
ax.set_title('Profit Margin by Sub-Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Profit Margin (%)', fontsize=11)
ax.set_ylabel('Sub-Category', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/charts/profit_margin_by_sub_category.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.7 — Customer Segment Analysis


In [ ]:
segment_analysis = (
    df.groupby('customer_segment', as_index=False)
    .agg(order_count=('order_id', 'count'), total_sales=('sales', 'sum'), mean_order_value=('sales', 'mean'))
    .sort_values('total_sales', ascending=False)
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [
    ('order_count', 'Order Count', 'Orders'),
    ('total_sales', 'Total Sales', 'Sales ($)'),
    ('mean_order_value', 'Mean Order Value', 'Sales ($)'),
]

for axis, (metric, title, ylabel) in zip(axes, metrics):
    bars = axis.bar(segment_analysis['customer_segment'], segment_analysis[metric], width=0.6, color=COLORS[:len(segment_analysis)])
    axis.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
    axis.set_title(title, fontsize=14, fontweight='bold')
    axis.set_xlabel('Customer Segment', fontsize=11)
    axis.set_ylabel(ylabel, fontsize=11)
    axis.tick_params(axis='both', labelsize=10)
    axis.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/charts/customer_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.8 — Discount vs Profit Scatter


In [ ]:
CORRELATION_RELEVANCE_THRESHOLD = 0.20
REGRESSION_POINT_COUNT = 100

scatter_df = df[['discount', 'profit_margin', 'category']].dropna().copy()
discount_margin_correlation = scatter_df['discount'].corr(scatter_df['profit_margin'])

plt.figure(figsize=(10, 5))
ax = plt.gca()
sns.scatterplot(
    data=scatter_df,
    x='discount',
    y='profit_margin',
    hue='category',
    palette=COLORS,
    alpha=0.5,
    edgecolors='none',
    s=60,
    ax=ax,
)

if abs(discount_margin_correlation) >= CORRELATION_RELEVANCE_THRESHOLD:
    regression_coefficients = np.polyfit(scatter_df['discount'], scatter_df['profit_margin'], 1)
    regression_x = np.linspace(scatter_df['discount'].min(), scatter_df['discount'].max(), REGRESSION_POINT_COUNT)
    regression_y = np.poly1d(regression_coefficients)(regression_x)
    ax.plot(regression_x, regression_y, color='black', linewidth=2, label='Regression Line')

ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Effect of Discount on Profit Margin', fontsize=14, fontweight='bold')
ax.set_xlabel('Discount', fontsize=11)
ax.set_ylabel('Profit Margin (%)', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
ax.legend(title='Category')
plt.tight_layout()
plt.savefig('outputs/charts/discount_vs_profit_margin.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.9 — Correlation Heatmap


In [ ]:
numeric_columns = ['sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'profit_margin', 'days_to_ship']
correlation_matrix = df[numeric_columns].corr()

plt.figure(figsize=(10, 5))
ax = plt.gca()
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.5,
    square=True,
    ax=ax,
)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Numeric Features', fontsize=11)
ax.set_ylabel('Numeric Features', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
plt.tight_layout()
plt.savefig('outputs/charts/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.10 — Monthly Sales by Category (stacked area chart)


In [ ]:
monthly_category_sales = df.pivot_table(index='month_year', columns='category', values='sales', aggfunc='sum').fillna(0)
monthly_category_sales.index = monthly_category_sales.index.astype(str)

plt.figure(figsize=(10, 5))
ax = monthly_category_sales.plot(kind='area', stacked=True, alpha=0.6, color=COLORS, ax=plt.gca())
ax.set_title('Sales Composition by Category Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Sales ($)', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/charts/monthly_sales_by_category.png', dpi=150, bbox_inches='tight')
plt.show()


## SECTION 5 — RFM CUSTOMER SEGMENTATION


In [ ]:
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm_df = (
    df.groupby('customer_name')
    .agg(
        recency=('order_date', lambda dates: (snapshot_date - dates.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('sales', 'sum'),
    )
    .reset_index()
)

rfm_df['recency_score'] = pd.qcut(rfm_df['recency'].rank(method='first'), q=4, labels=[4, 3, 2, 1]).astype(str)
rfm_df['frequency_score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(str)
rfm_df['monetary_score'] = pd.qcut(rfm_df['monetary'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(str)
rfm_df['rfm_score'] = rfm_df['recency_score'] + rfm_df['frequency_score'] + rfm_df['monetary_score']

rfm_df['rfm_segment'] = np.select(
    [
        rfm_df['rfm_score'] == '444',
        rfm_df['rfm_score'].str.startswith('4'),
        rfm_df['rfm_score'].str.startswith('1'),
    ],
    ['Champion', 'Loyal Customer', 'At Risk'],
    default='Regular',
)

print('Top 10 customers by monetary value:')
display(rfm_df.sort_values('monetary', ascending=False).head(10))


## SECTION 6 — SQL ANALYSIS


In [ ]:
conn = sqlite3.connect(':memory:')
orders_sql = df.copy()
orders_sql['order_date'] = orders_sql['order_date'].dt.strftime('%Y-%m-%d')
orders_sql['ship_date'] = orders_sql['ship_date'].dt.strftime('%Y-%m-%d')
orders_sql['month_year'] = orders_sql['month_year'].astype(str)
orders_sql.to_sql('orders', conn, index=False, if_exists='replace')

queries = {
    'query_1_top_10_products_by_revenue': """
        SELECT 
            product_name,
            COUNT(order_id) AS total_orders,
            ROUND(SUM(sales), 2) AS total_revenue,
            ROUND(SUM(profit), 2) AS total_profit,
            ROUND(AVG(profit / sales * 100), 2) AS avg_margin_pct
        FROM orders
        GROUP BY product_name
        ORDER BY total_revenue DESC
        LIMIT 10;
    """,
    'query_2_monthly_revenue_profit_summary': """
        SELECT
            strftime('%Y-%m', order_date) AS month,
            ROUND(SUM(sales), 2) AS monthly_revenue,
            ROUND(SUM(profit), 2) AS monthly_profit,
            ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_pct
        FROM orders
        GROUP BY month
        ORDER BY month;
    """,
    'query_3_revenue_profit_by_customer_segment': """
        SELECT
            customer_segment,
            COUNT(DISTINCT customer_name) AS unique_customers,
            COUNT(order_id) AS total_orders,
            ROUND(SUM(sales), 2) AS total_revenue,
            ROUND(AVG(sales), 2) AS avg_order_value,
            ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_pct
        FROM orders
        GROUP BY customer_segment
        ORDER BY total_revenue DESC;
    """,
    'query_4_top_10_states_by_average_profit_margin': """
        SELECT
            state,
            COUNT(order_id) AS orders,
            ROUND(SUM(sales), 2) AS total_sales,
            ROUND(AVG(profit / sales * 100), 2) AS avg_profit_margin_pct
        FROM orders
        GROUP BY state
        HAVING orders >= 10
        ORDER BY avg_profit_margin_pct DESC
        LIMIT 10;
    """,
    'query_5_loss_making_products': """
        SELECT
            product_name,
            category,
            COUNT(order_id) AS times_ordered,
            ROUND(SUM(sales), 2) AS total_sales,
            ROUND(SUM(profit), 2) AS total_profit
        FROM orders
        GROUP BY product_name, category
        HAVING total_profit < 0
        ORDER BY total_profit ASC;
    """,
}

sql_results = {}
for query_name, query_text in queries.items():
    result_df = pd.read_sql_query(query_text, conn)
    sql_results[query_name] = result_df
    result_df.to_csv(f'outputs/reports/{query_name}.csv', index=False)
    print()
    print(query_name.replace('_', ' ').title())
    display(result_df)

conn.close()


## SECTION 7 — BUSINESS INSIGHTS

The following cell generates six numbered business insights directly from the cleaned dataset. This keeps the recommendations specific to the actual CSV used in the project while preserving a professional markdown-style output.


In [ ]:
from IPython.display import Markdown, display

LOW_MARGIN_THRESHOLD = 0
STRONG_MARGIN_THRESHOLD = 15

lowest_margin_subcategory = subcategory_margin.iloc[0]
highest_revenue_region = regional_performance.iloc[0]
highest_revenue_category = category_performance.sort_values('sales', ascending=False).iloc[0]
top_revenue_product = top_products.sort_values('sales', ascending=False).iloc[0]
segment_leader = segment_analysis.iloc[0]
loss_making_products = sql_results['query_5_loss_making_products']
loss_product_text = (
    f"{loss_making_products.iloc[0]['product_name']} lost ${abs(loss_making_products.iloc[0]['total_profit']):,.2f}"
    if not loss_making_products.empty
    else 'no products had negative total profit'
)

insights_markdown = f'''
1. Finding: The **{lowest_margin_subcategory['sub_category']}** sub-category has the weakest average profit margin at **{lowest_margin_subcategory['profit_margin']:.2f}%**. Recommendation: Review discounting, supplier costs, and shipping charges for this sub-category before increasing promotional activity.

2. Finding: The **{highest_revenue_region['region']}** region generates the highest sales at **${highest_revenue_region['sales']:,.2f}**, with profit of **${highest_revenue_region['profit']:,.2f}**. Recommendation: Protect this region with strong inventory availability and use its best-performing tactics as a benchmark for lower-performing regions.

3. Finding: The **{highest_revenue_category.name}** category leads revenue with **${highest_revenue_category['sales']:,.2f}** in sales and **${highest_revenue_category['profit']:,.2f}** in profit. Recommendation: Prioritize merchandising and cross-selling around this category while monitoring whether growth is also improving margin.

4. Finding: The top revenue product is **{top_revenue_product['product_name']}**, producing **${top_revenue_product['sales']:,.2f}** in sales. Recommendation: Keep this product visible in campaigns, but validate its margin before relying on it as a profit driver.

5. Finding: The **{segment_leader['customer_segment']}** customer segment contributes the most revenue at **${segment_leader['total_sales']:,.2f}** with an average order value of **${segment_leader['mean_order_value']:,.2f}**. Recommendation: Build targeted retention offers for this segment and test upsell bundles to raise order value further.

6. Finding: The loss-making product review shows that **{loss_product_text}**. Recommendation: Investigate these SKUs for excessive discounting, high shipping costs, or low pricing, and consider repricing or discontinuing persistent loss-makers.
'''

display(Markdown(insights_markdown))
